# BERT WSI

Linear, function-free Word Sense Induction experiment using RuBERT contextual embeddings for the target word.

In [ ]:
import re

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

## Load Data

In [ ]:
df = pd.read_csv("data/train/train.csv", sep="\t")
df = df.drop(columns=["predict_sense_id"], errors="ignore")

print(f"Rows: {len(df)}")
print(f"Words: {df['word'].nunique()}")
print(f"Missing positions: {df['positions'].isna().sum()}")
df.head()

## Validation Split

In [ ]:
random_state = 42
valid_frac = 0.2

rng = np.random.RandomState(random_state)
train_idx = []
valid_idx = []

for _, word_df in df.groupby("word"):
    for _, sense_df in word_df.groupby("gold_sense_id"):
        indices = sense_df.index.to_numpy().copy()
        rng.shuffle(indices)

        if len(indices) < 2:
            train_idx.extend(indices.tolist())
            continue

        n_valid = max(1, int(round(len(indices) * valid_frac)))
        n_valid = min(n_valid, len(indices) - 1)

        valid_idx.extend(indices[:n_valid].tolist())
        train_idx.extend(indices[n_valid:].tolist())

train_df = df.loc[sorted(train_idx)].copy()
valid_df = df.loc[sorted(valid_idx)].copy()

print(f"Train rows: {len(train_df)}")
print(f"Valid rows: {len(valid_df)}")

## Resolve Target Positions

In [ ]:
target_token_pattern = re.compile(r"[A-Za-z\u0410-\u042f\u0430-\u044f\u0401\u0451-]+")

for split_name, split_df in [("train", train_df), ("valid", valid_df)]:
    resolved_positions = []
    unresolved_context_ids = []

    for row in split_df.itertuples():
        position_value = np.nan
        positions_str = "" if pd.isna(row.positions) else str(row.positions)

        for chunk in positions_str.split(","):
            chunk = chunk.strip()
            if "-" not in chunk:
                continue
            start_text, end_text = chunk.split("-", 1)
            try:
                start_char = int(start_text)
                end_char = int(end_text)
                position_value = f"{start_char}-{end_char}"
                break
            except ValueError:
                pass

        if pd.isna(position_value):
            text_lower = str(row.context).lower()
            word_lower = str(row.word).lower()
            exact_matches = [match.span() for match in re.finditer(re.escape(word_lower), text_lower)]
            if len(exact_matches) == 1:
                start_char, end_char = exact_matches[0]
                position_value = f"{start_char}-{end_char}"

        if pd.isna(position_value):
            word_lower = str(row.word).lower()
            token_matches = []
            for match in target_token_pattern.finditer(str(row.context)):
                token = match.group(0)
                prefix_len = 0
                for token_char, word_char in zip(token.lower(), word_lower):
                    if token_char != word_char:
                        break
                    prefix_len += 1
                min_prefix = max(3, min(len(token), len(word_lower)) - 2)
                if prefix_len >= min_prefix:
                    token_matches.append(match.span())

            if len(token_matches) == 1:
                start_char, end_char = token_matches[0]
                position_value = f"{start_char}-{end_char}"

        if pd.isna(position_value):
            unresolved_context_ids.append(row.context_id)

        resolved_positions.append(position_value)

    split_df["resolved_positions"] = resolved_positions
    failed_rows = split_df[split_df["resolved_positions"].isna()][["context_id", "word", "context"]].copy()
    bert_df = split_df[split_df["resolved_positions"].notna()].copy()
    bert_df["positions"] = bert_df["resolved_positions"]
    bert_df = bert_df.drop(columns=["resolved_positions"])

    if split_name == "train":
        train_failed = failed_rows
        train_bert_df = bert_df
    else:
        valid_failed = failed_rows
        valid_bert_df = bert_df

print(f"Train unresolved rows: {len(train_failed)}")
print(f"Valid unresolved rows: {len(valid_failed)}")
print(f"Train BERT rows: {len(train_bert_df)}")
print(f"Valid BERT rows: {len(valid_bert_df)}")

## Load BERT

In [ ]:
MODEL_NAME = "cointegrated/rubert-tiny2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(random_state)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_model.eval()

DEVICE

## Build Train Embeddings

In [ ]:
train_vectors = []

for row in tqdm(train_bert_df.itertuples(index=False), total=len(train_bert_df)):
    start_char = None
    end_char = None
    for chunk in str(row.positions).split(","):
        chunk = chunk.strip()
        if "-" not in chunk:
            continue
        start_text, end_text = chunk.split("-", 1)
        start_char = int(start_text)
        end_char = int(end_text)
        break

    encoded = tokenizer(
        str(row.context),
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=512,
    )
    offset_mapping = encoded.pop("offset_mapping")[0].tolist()
    model_inputs = {key: value.to(DEVICE) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = bert_model(**model_inputs)

    hidden = outputs.last_hidden_state[0]
    token_indices = []

    for token_idx, (token_start, token_end) in enumerate(offset_mapping):
        if token_end <= token_start:
            continue
        if token_end <= start_char:
            continue
        if token_start >= end_char:
            continue
        token_indices.append(token_idx)

    if len(token_indices) == 0:
        token_indices = [1]

    train_vectors.append(hidden[token_indices].mean(dim=0).detach().cpu().numpy())

x_train = np.vstack(train_vectors)
x_train.shape

## Build Validation Embeddings

In [ ]:
valid_vectors = []

for row in tqdm(valid_bert_df.itertuples(index=False), total=len(valid_bert_df)):
    start_char = None
    end_char = None
    for chunk in str(row.positions).split(","):
        chunk = chunk.strip()
        if "-" not in chunk:
            continue
        start_text, end_text = chunk.split("-", 1)
        start_char = int(start_text)
        end_char = int(end_text)
        break

    encoded = tokenizer(
        str(row.context),
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=512,
    )
    offset_mapping = encoded.pop("offset_mapping")[0].tolist()
    model_inputs = {key: value.to(DEVICE) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = bert_model(**model_inputs)

    hidden = outputs.last_hidden_state[0]
    token_indices = []

    for token_idx, (token_start, token_end) in enumerate(offset_mapping):
        if token_end <= token_start:
            continue
        if token_end <= start_char:
            continue
        if token_start >= end_char:
            continue
        token_indices.append(token_idx)

    if len(token_indices) == 0:
        token_indices = [1]

    valid_vectors.append(hidden[token_indices].mean(dim=0).detach().cpu().numpy())

x_valid = np.vstack(valid_vectors)
x_valid.shape

## Train Logistic Regression Per Word

In [ ]:
predictions = pd.Series(index=valid_bert_df.index, dtype="object")

for word in sorted(valid_bert_df["word"].unique()):
    train_mask = (train_bert_df["word"] == word).to_numpy()
    valid_mask = (valid_bert_df["word"] == word).to_numpy()
    y_train = train_bert_df.loc[train_mask, "gold_sense_id"].astype(str)

    if y_train.nunique() == 1:
        pred = np.repeat(y_train.iloc[0], valid_mask.sum())
    else:
        clf = LogisticRegression(max_iter=1000, random_state=random_state)
        clf.fit(x_train[train_mask], y_train)
        pred = clf.predict(x_valid[valid_mask])

    predictions.loc[valid_bert_df.index[valid_mask]] = pred

scored_df = valid_bert_df[["word", "gold_sense_id"]].copy()
scored_df["prediction"] = predictions.astype(str)
scored_df["gold_sense_id"] = scored_df["gold_sense_id"].astype(str)

run_word_scores = []
for word, word_df in scored_df.groupby("word"):
    ari = adjusted_rand_score(word_df["gold_sense_id"], word_df["prediction"])
    run_word_scores.append((word, ari, len(word_df)))

bert_word_scores_df = pd.DataFrame(run_word_scores, columns=["word", "ari", "n_valid"])
macro_ari = bert_word_scores_df["ari"].mean()
weighted_ari = np.average(bert_word_scores_df["ari"], weights=bert_word_scores_df["n_valid"])

bert_runs_df = pd.DataFrame(
    [(
        "bert_target_embedding",
        random_state,
        macro_ari,
        weighted_ari,
        len(train_failed),
        len(valid_failed),
    )],
    columns=["model", "random_state", "macro_ari", "weighted_ari", "train_unresolved", "valid_unresolved"],
)

bert_summary_df = (
    bert_runs_df.groupby("model", as_index=False)
    .agg(
        macro_ari_mean=("macro_ari", "mean"),
        macro_ari_std=("macro_ari", "std"),
        weighted_ari_mean=("weighted_ari", "mean"),
        weighted_ari_std=("weighted_ari", "std"),
    )
    .sort_values("macro_ari_mean", ascending=False)
)
bert_summary_df[["macro_ari_std", "weighted_ari_std"]] = bert_summary_df[["macro_ari_std", "weighted_ari_std"]].fillna(0.0)

bert_summary_df